# Autonomous AI Data Scientist (AADS) — Google Colab Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uday0214/IOTAxEngimaHackathon/blob/main/notebooks/colab_training.ipynb)

**Competition**: Utkrishta 2026 AI/ML Hackathon — Challenge 1 (Budget-Constrained Churn Prediction)

This interactive Google Colab notebook executes the **Autonomous AI Data Scientist** framework:
- **Dual-Brain Supervisor**: Google Gemini 2.5 Flash LLM + Deterministic Mathematical Inference Engine
- **Budget Invariant**: Strict $\le 15$ model fits enforced by atomic `BudgetGuard`
- **Hardware Acceleration**: Automatic NVIDIA GPU detection and CatBoost CUDA acceleration
- **Data Integrity Guard**: SHA-256 cryptographic hash and index verification against data leakage
- **End-to-End Pipeline**: Zero-cost profiling, 15-run feedback loop, sealed holdout evaluation, and submission export

## Step 1: Verify Hardware Accelerator (GPU Check)

Ensure your Google Colab runtime is set to GPU:
1. Go to the menu: **Runtime** -> **Change runtime type**
2. Hardware accelerator: Select **T4 GPU** (or A100 / V100)
3. Click **Save**

In [ ]:
!nvidia-smi

## Step 2: Clone Repository & Setup Working Directory

In [ ]:
import os
import sys

# Clone repository on multi-model-tournament branch or switch to it
repo_name = "IOTAxEngimaHackathon"
if not os.path.exists(repo_name):
    !git clone -b multi-model-tournament https://github.com/uday0214/IOTAxEngimaHackathon.git
    %cd IOTAxEngimaHackathon
else:
    %cd IOTAxEngimaHackathon
    !git fetch origin
    !git checkout multi-model-tournament
    !git pull origin multi-model-tournament

sys.path.insert(0, os.getcwd())
print(f"Current working directory: {os.getcwd()}")

## Step 3: Install Required Dependencies

In [ ]:
!pip install -q -r requirements.txt
print("Dependencies installed successfully!")

## Step 4: Configure & Verify Supervisor LLM Model (Free Model Selection)

Choose any model from OpenRouter (free or paid) or Google Gemini. Use the interactive form below to select or type your desired model.

### Suggested Models:
- `nvidia/nemotron-3-ultra-550b-a55b:free` (Free, High Capacity)
- `meta-llama/llama-3.3-70b-instruct`
- `deepseek/deepseek-chat`
- `google/gemini-2.5-flash`
- `anthropic/claude-3.5-sonnet`

In [ ]:
# @title Supervisor LLM Configuration & Live Verification
# @markdown Choose your provider and freely type or select any model name:

LLM_PROVIDER = "openrouter"  # @param ["openrouter", "gemini", "mock"]
SELECTED_MODEL = "nvidia/nemotron-3-ultra-550b-a55b:free"  # @param ["nvidia/nemotron-3-ultra-550b-a55b:free", "google/gemini-2.5-flash", "meta-llama/llama-3.3-70b-instruct", "deepseek/deepseek-chat", "anthropic/claude-3.5-sonnet"] {allow-input: true}

import os
from src.utils.environment import get_openrouter_api_key, get_gemini_api_key
from src.supervisor.llm_client import SupervisorLLMClient

os.environ["LLM_PROVIDER"] = LLM_PROVIDER.strip().lower()
if LLM_PROVIDER == "openrouter":
    os.environ["OPENROUTER_MODEL"] = SELECTED_MODEL.strip()
    key = get_openrouter_api_key()
    if not key:
        import getpass
        key = getpass.getpass("Enter your OPENROUTER_API_KEY (or leave blank for mock mode): ").strip()
    if key:
        os.environ["OPENROUTER_API_KEY"] = key
elif LLM_PROVIDER == "gemini":
    os.environ["GEMINI_MODEL"] = SELECTED_MODEL.strip()
    key = get_gemini_api_key()
    if not key:
        import getpass
        key = getpass.getpass("Enter your GEMINI_API_KEY (or leave blank for mock mode): ").strip()
    if key:
        os.environ["GEMINI_API_KEY"] = key

# Verify connection and show verification output
client = SupervisorLLMClient()
check = client.verify_connection()

print("=" * 65)
print("SUPERVISOR MODEL VERIFICATION REPORT")
print("=" * 65)
print(f"Status:    {check.get('status')}")
print(f"Provider:  {check.get('provider').upper()}")
print(f"Model:     {check.get('model')}")
print(f"Latency:   {check.get('latency_sec', 0.0)}s")
print(f"Message:   {check.get('message', check.get('error', 'Ready'))}")
print("=" * 65)

if check.get('status') == 'READY':
    print("\u2705 Model successfully loaded and verified! Ready for autonomous training.")
elif check.get('status') == 'MOCK_MODE':
    print("\u2139\ufe0f Running in offline mock supervisor mode.")
else:
    print("\u26a0\ufe0f Verification probe failed. Check API key or selected model name.")

## Step 5: Data Preparation & Cryptographic Manifest Generation

Prepares the 80% train-val and 20% sealed holdout partitions and generates the immutable SHA-256 fingerprint registry (`artifacts/dataset_manifest.json`).

In [ ]:
!python scripts/prepare_data.py

## Step 6: Launch the 15-Run Autonomous Training Loop

Runs the end-to-end autonomous optimization loop with GPU acceleration (`--device auto`) and live model fitting (`--live`).
Strictly capped at 15 experiments by the atomic `BudgetGuard`.

In [ ]:
!python scripts/run_autonomous_loop.py --live --device auto --clean-ledger

## Step 7: Telemetry Inspection & Interactive Visualization

Load the JSONL experiment ledger and visualize:
1. **ROC-AUC & PR-AUC Progression** across the 15 runs
2. **Overfitting Index & Loss Trajectory** (Train vs Validation Loss)
3. **Bayesian Hyperparameter Search Convergence**

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

# Load experiment ledger
runs = []
with open("artifacts/experiment_ledger.jsonl", "r") as f:
    for line in f:
        if line.strip():
            runs.append(json.loads(line))

ledger_df = pd.DataFrame([
    {
        "Run": r["experiment_id"],
        "Model": r["hyperparameters"].get("model_family", "catboost"),
        "ROC_AUC": r["metrics"]["oof_roc_auc"],
        "PR_AUC": r["metrics"]["oof_pr_auc"],
        "Train_Loss": r["metrics"]["final_train_loss"],
        "Val_Loss": r["metrics"]["final_val_loss"],
        "Depth": r["hyperparameters"].get("depth"),
        "Learning_Rate": r["hyperparameters"].get("learning_rate"),
        "L2_Leaf_Reg": r["hyperparameters"].get("l2_leaf_reg"),
    }
    for r in runs
])

print("=== EXPERIMENT LEDGER SUMMARY ===")
display(ledger_df)

# Plot Metric Trajectories
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(ledger_df["Run"], ledger_df["ROC_AUC"], marker="o", label="OOF ROC-AUC", color="#1f77b4", lw=2)
axes[0].plot(ledger_df["Run"], ledger_df["PR_AUC"], marker="s", label="OOF PR-AUC", color="#2ca02c", lw=2)
axes[0].set_title("Validation Metrics Progression (15 Runs)", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Experiment Run")
axes[0].set_ylabel("Metric Score")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend()

axes[1].plot(ledger_df["Run"], ledger_df["Train_Loss"], marker="o", label="Final Train Loss", color="#ff7f0e", lw=2)
axes[1].plot(ledger_df["Run"], ledger_df["Val_Loss"], marker="x", label="Final Val Loss", color="#d62728", lw=2)
axes[1].set_title("Generalization Gap & Loss Convergence", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Experiment Run")
axes[1].set_ylabel("Log Loss")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend()

plt.tight_layout()
plt.show()

# Render Supervisor Final Analytical Report (Visualizable Data)
from IPython.display import display, Markdown, HTML
report_file = Path("artifacts/SUPERVISOR_FINAL_REPORT.md")
if report_file.exists():
    print("\n=======================================================")
    print("SUPERVISOR FINAL POST-MORTEM REPORT")
    print("=======================================================")
    with open(report_file, encoding="utf-8") as f:
        display(Markdown(f.read()))

# Render Interactive Dashboard Link/Preview
dashboard_file = Path("artifacts/telemetry/trajectory_dashboard.html")
if dashboard_file.exists():
    print("\nTelemetry Dashboard available at: artifacts/telemetry/trajectory_dashboard.html")

## Step 8: Evaluate Sealed Holdout & Export Competition Submission

Following the completion of Experiment 15, unseal the 20% holdout set and generate `artifacts/submission.csv`.

In [ ]:
!python scripts/evaluate_holdout.py
!python scripts/export_submission.py

## Step 9: Download Artifacts or Persist to Google Drive

Bundle the experiment ledger, champion model, holdout metrics, and submission file into a downloadable archive.

In [ ]:
import shutil
from pathlib import Path

# Archive the artifacts folder
shutil.make_archive("aads_artifacts", "zip", "artifacts")
print("Artifacts archived to aads_artifacts.zip")

# Optional: Trigger browser download in Google Colab
try:
    from google.colab import files
    files.download("aads_artifacts.zip")
    files.download("artifacts/submission.csv")
except Exception as e:
    print(f"Note: If running locally or download prompt blocked, file is available at aads_artifacts.zip: {e}")

## Step 10: (Optional) Push Trained Artifacts Directly to GitHub

Push the completed experiment ledger, champion model, and holdout results directly to branch `uday` on GitHub.

### Instructions:
1. Store your GitHub Personal Access Token (PAT) in Colab Secrets under `GITHUB_TOKEN`, OR enter it when prompted below.
2. Run the cell to commit and push.

In [ ]:
import os
import getpass

# 1. Try to get GitHub token from Colab secrets, or prompt
github_token = None
try:
    from google.colab import userdata
    github_token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

if not github_token:
    github_token = getpass.getpass("Enter your GitHub Personal Access Token (or press Enter to skip): ").strip()

if github_token:
    current_branch = os.popen("git rev-parse --abbrev-ref HEAD").read().strip() or "main"
    !git config --global user.name "AADS Colab Runner"
    !git config --global user.email "aads-colab@hackathon.local"
    !git remote set-url origin https://{github_token}@github.com/uday0214/IOTAxEngimaHackathon.git
    !git add artifacts/
    !git commit -m "feat(colab): completed 15-run autonomous training and exported champion model"
    !git push origin {current_branch}
    print(f"Successfully committed and pushed all trained artifacts to GitHub on branch '{current_branch}'!")
else:
    print("Skipped GitHub push. Download 'aads_artifacts.zip' from Step 9 to save locally.")